**SQL Task – Cyber Threat Data Analysis**

This notebook demonstrates the use of SQL queries on a cybersecurity threat dataset. The dataset was generated using AI prompting and analyzed using SQL via SQLite in Google Colab. All SQL operations are performed based on structured prompts.

**Prompt Used for SQL Environment Setup**

"Set up an SQL environment in Google Colab using SQLite to perform SQL queries on a cybersecurity dataset."

In [3]:
import sqlite3
import pandas as pd


In [12]:
import pandas as pd
import sqlite3
import numpy as np

# Generate synthetic cybersecurity threat data
np.random.seed(42)
num_entries = 300

threat_types = ['Malware', 'Phishing', 'DDoS', 'Ransomware', 'Insider Threat', 'SQL Injection']
severities = ['Low', 'Medium', 'High', 'Critical']
statuses = ['Detected', 'Mitigated', 'Active', 'False Positive']

data = {
    'threat_id': [f'THREAT_{i:04d}' for i in range(num_entries)],
    'timestamp': pd.to_datetime(pd.date_range(start='2023-01-01', periods=num_entries, freq='h')).strftime('%Y-%m-%d %H:%M:%S'),
    'threat_type': np.random.choice(threat_types, num_entries),
    'severity': np.random.choice(severities, num_entries, p=[0.4, 0.3, 0.2, 0.1]),
    'source_ip': [f'192.168.1.{np.random.randint(1, 255)}' for _ in range(num_entries)],
    'target_ip': [f'10.0.0.{np.random.randint(1, 255)}' for _ in range(num_entries)],
    'status': np.random.choice(statuses, num_entries),
    'description': [f'Suspicious activity related to {t}' for t in np.random.choice(threat_types, num_entries)],
}

df = pd.DataFrame(data)

# Save the synthetic data to a CSV file
df.to_csv("/content/synthetic_cyber_threats_300.csv", index=False)

print("Synthetic dataset generated and saved to /content/synthetic_cyber_threats_300.csv")

# Create SQLite database
conn = sqlite3.connect("cyber_threats.db")

# Load DataFrame into SQL table
df.to_sql("cyber_threats", conn, if_exists="replace", index=False)

print("Dataset loaded into SQLite database.")

Synthetic dataset generated and saved to /content/synthetic_cyber_threats_300.csv
Dataset loaded into SQLite database.


In [13]:

query = "SELECT * FROM cyber_threats LIMIT 5;"
pd.read_sql(query, conn)

,threat_id,timestamp,threat_type,severity,source_ip,target_ip,status,description
0,THREAT_0000,2023-01-01 00:00:00,Ransomware,Low,192.168.1.141,10.0.0.6,Active,Suspicious activity related to Phishing
1,THREAT_0001,2023-01-01 01:00:00,Insider Threat,Critical,192.168.1.46,10.0.0.180,Active,Suspicious activity related to Ransomware
2,THREAT_0002,2023-01-01 02:00:00,DDoS,Critical,192.168.1.35,10.0.0.26,Detected,Suspicious activity related to SQL Injection
3,THREAT_0003,2023-01-01 03:00:00,Insider Threat,Low,192.168.1.253,10.0.0.64,Mitigated,Suspicious activity related to DDoS
4,THREAT_0004,2023-01-01 04:00:00,Insider Threat,Low,192.168.1.134,10.0.0.226,Detected,Suspicious activity related to Malware


In [14]:
query = "SELECT * FROM cyber_threats;"
pd.read_sql(query, conn).head()


,threat_id,timestamp,threat_type,severity,source_ip,target_ip,status,description
0,THREAT_0000,2023-01-01 00:00:00,Ransomware,Low,192.168.1.141,10.0.0.6,Active,Suspicious activity related to Phishing
1,THREAT_0001,2023-01-01 01:00:00,Insider Threat,Critical,192.168.1.46,10.0.0.180,Active,Suspicious activity related to Ransomware
2,THREAT_0002,2023-01-01 02:00:00,DDoS,Critical,192.168.1.35,10.0.0.26,Detected,Suspicious activity related to SQL Injection
3,THREAT_0003,2023-01-01 03:00:00,Insider Threat,Low,192.168.1.253,10.0.0.64,Mitigated,Suspicious activity related to DDoS
4,THREAT_0004,2023-01-01 04:00:00,Insider Threat,Low,192.168.1.134,10.0.0.226,Detected,Suspicious activity related to Malware


In [15]:
query = "SELECT COUNT(*) AS total_attacks FROM cyber_threats;"
pd.read_sql(query, conn)

,total_attacks
0,300


In [19]:
query = """
SELECT threat_type, COUNT(*) AS attack_count
FROM cyber_threats
GROUP BY threat_type
ORDER BY attack_count DESC;
"""
pd.read_sql(query, conn)

,threat_type,attack_count
0,Ransomware,64
1,Malware,57
2,DDoS,52
3,SQL Injection,46
4,Phishing,42
5,Insider Threat,39


In [22]:
# Map severity levels to numeric scores
severity_mapping = {'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4}
df['severity_score'] = df['severity'].map(severity_mapping)

# Update the SQLite table with the new severity_score column
# This will replace the existing table with the updated DataFrame
df.to_sql('cyber_threats', conn, if_exists='replace', index=False)

print("Added 'severity_score' column and updated the database.")

Added 'severity_score' column and updated the database.


In [21]:
query = """
SELECT
    threat_type,
    AVG(severity_score) AS average_severity_score
FROM
    cyber_threats
GROUP BY
    threat_type
ORDER BY
    average_severity_score DESC;
"""

pd.read_sql(query, conn)

,threat_type,average_severity_score
0,Phishing,2.285714
1,Insider Threat,2.128205
2,DDoS,2.019231
3,Ransomware,1.859375
4,Malware,1.824561
5,SQL Injection,1.760870


In [24]:
query = """
SELECT severity, COUNT(*) AS count
FROM cyber_threats
GROUP BY severity;
"""
pd.read_sql(query, conn)


,severity,count
0,Critical,35
1,High,46
2,Low,128
3,Medium,91


In [25]:
query = """
SELECT *
FROM cyber_threats
WHERE severity = 'High';
"""
pd.read_sql(query, conn).head()


,threat_id,timestamp,threat_type,severity,source_ip,target_ip,status,description,severity_score
0,THREAT_0013,2023-01-01 13:00:00,Insider Threat,High,192.168.1.30,10.0.0.181,Active,Suspicious activity related to DDoS,3
1,THREAT_0014,2023-01-01 14:00:00,Phishing,High,192.168.1.242,10.0.0.150,Mitigated,Suspicious activity related to Ransomware,3
2,THREAT_0028,2023-01-02 04:00:00,Malware,High,192.168.1.85,10.0.0.55,Mitigated,Suspicious activity related to DDoS,3
3,THREAT_0032,2023-01-02 08:00:00,Ransomware,High,192.168.1.86,10.0.0.243,False Positive,Suspicious activity related to SQL Injection,3
4,THREAT_0037,2023-01-02 13:00:00,DDoS,High,192.168.1.161,10.0.0.152,Mitigated,Suspicious activity related to Malware,3


In [28]:
import numpy as np

# Define a list of plausible source countries for synthetic data
countries = ['USA', 'China', 'Russia', 'Germany', 'India', 'Brazil', 'UK', 'Japan', 'Canada', 'Australia']

# Assign random countries to the DataFrame
df['source_country'] = np.random.choice(countries, len(df))

# Update the SQLite table with the new 'source_country' column
# This will replace the existing table with the updated DataFrame
df.to_sql('cyber_threats', conn, if_exists='replace', index=False)

# Now, execute the query with the new 'source_country' column
query = """
SELECT source_country, COUNT(*) AS total_attacks
FROM cyber_threats
GROUP BY source_country
ORDER BY total_attacks DESC;
"""
pd.read_sql(query, conn)

,source_country,total_attacks
0,China,35
1,Brazil,35
2,Germany,33
3,USA,30
4,Australia,30
5,Russia,29
6,Canada,29
7,UK,27
8,Japan,27
9,India,25


In [30]:
import numpy as np

# Define a list of plausible devices for synthetic data
devices = ['Server', 'Workstation', 'Laptop', 'Mobile', 'IoT Device']

# Assign random devices to the DataFrame
df['device_affected'] = np.random.choice(devices, len(df))

# Update the SQLite table with the new 'device_affected' column
# This will replace the existing table with the updated DataFrame
df.to_sql('cyber_threats', conn, if_exists='replace', index=False)

query = """
SELECT device_affected, COUNT(*) AS attack_count
FROM cyber_threats
GROUP BY device_affected
ORDER BY attack_count DESC;
"""
pd.read_sql(query, conn)

,device_affected,attack_count
0,Workstation,66
1,Mobile,63
2,IoT Device,62
3,Server,61
4,Laptop,48


In [33]:
query = """
SELECT
    status,
    COUNT(*) AS count
FROM cyber_threats
GROUP BY status
ORDER BY count DESC;
"""
pd.read_sql(query, conn)

,status,count
0,Mitigated,81
1,Detected,80
2,Active,77
3,False Positive,62


In [8]:
import pandas as pd
import numpy as np
import sqlite3

# Regenerate the base synthetic data
np.random.seed(42)
num_entries = 300
threat_types = ['Malware', 'Phishing', 'DDoS', 'Ransomware', 'Insider Threat', 'SQL Injection']
severities = ['Low', 'Medium', 'High', 'Critical']
statuses = ['Detected', 'Mitigated', 'Active', 'False Positive']

data = {
    'threat_id': [f'THREAT_{i:04d}' for i in range(num_entries)],
    'timestamp': pd.date_range(start='2023-01-01', periods=num_entries, freq='h').strftime('%Y-%m-%d %H:%M:%S'),
    'threat_type': np.random.choice(threat_types, num_entries),
    'severity': np.random.choice(severities, num_entries),
    'status': np.random.choice(statuses, num_entries)
}
df = pd.DataFrame(data)

# Define and add target systems
systems = ['Financial', 'Human Resources', 'Customer Database', 'Infrastructure', 'External Web', 'Internal Research']
df['target_system'] = np.random.choice(systems, len(df))

# Connect and update the database
conn = sqlite3.connect('cyber_threats.db')
df.to_sql('cyber_threats', conn, if_exists='replace', index=False)

query = """
SELECT
    target_system,
    COUNT(*) AS count
FROM cyber_threats
GROUP BY target_system
ORDER BY count DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,target_system,count
0,Internal Research,59
1,Human Resources,52
2,Financial,51
3,Customer Database,51
4,External Web,50
5,Infrastructure,37


In [10]:
cursor = conn.cursor()
cursor.execute("PRAGMA table_info(cyber_threats)")
print(cursor.fetchall())

[(0, 'threat_id', 'TEXT', 0, None, 0), (1, 'timestamp', 'TEXT', 0, None, 0), (2, 'threat_type', 'TEXT', 0, None, 0), (3, 'severity', 'TEXT', 0, None, 0), (4, 'status', 'TEXT', 0, None, 0), (5, 'target_system', 'TEXT', 0, None, 0)]


In [12]:
query = """
SELECT AVG(Severity) AS avg_impact
FROM cyber_threats;
"""

pd.read_sql(query, conn)

,avg_impact
0,0.0


**Explanation**

The average impact level initially returned 0.0 because the impact_level column contained categorical text values rather than numeric data. SQL aggregate functions such as AVG require numeric input. To resolve this, the impact levels were mapped to numeric values and reloaded into the database, allowing accurate calculation of the average impact level.

In [15]:
query = "PRAGMA table_info(cyber_threats);"
pd.read_sql(query, conn)


,cid,name,type,notnull,dflt_value,pk
0,0,threat_id,TEXT,0,None,0
1,1,timestamp,TEXT,0,None,0
2,2,threat_type,TEXT,0,None,0
3,3,severity,TEXT,0,None,0
4,4,status,TEXT,0,None,0
5,5,target_system,TEXT,0,None,0


In [67]:
print(pd.read_sql("PRAGMA table_info(cyber_threats);", conn))

   cid           name  type  notnull dflt_value  pk
0    0      threat_id  TEXT        0       None   0
1    1      timestamp  TEXT        0       None   0
2    2    threat_type  TEXT        0       None   0
3    3       severity  TEXT        0       None   0
4    4         status  TEXT        0       None   0
5    5  target_system  TEXT        0       None   0


In [72]:
query = """
SELECT severity, COUNT(*) AS count
FROM cyber_threats
GROUP BY severity;
"""
pd.read_sql(query, conn)

,severity,count
0,Critical,82
1,High,61
2,Low,75
3,Medium,82


In [76]:
impact_mapping = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Critical": 4
}

df['severity_numeric'] = df['severity'].map(impact_mapping)

In [77]:
df.to_sql("cyber_threats", conn, if_exists="replace", index=False)

300

In [78]:
query = """
SELECT AVG(severity_numeric) AS avg_impact
FROM cyber_threats;
"""
pd.read_sql(query, conn)


,avg_impact
0,2.5


In [81]:
query = """
SELECT threat_type, AVG(severity_numeric) AS avg_severity
FROM cyber_threats
GROUP BY threat_type
HAVING AVG(severity_numeric) > (
    SELECT AVG(severity_numeric)
    FROM cyber_threats
);
"""

pd.read_sql(query, conn)

,threat_type,avg_severity
0,DDoS,2.519231
1,Insider Threat,2.512821
2,Phishing,2.595238
3,Ransomware,2.562500


In [83]:
query = """
SELECT threat_type, COUNT(*) AS total_attacks
FROM cyber_threats
GROUP BY threat_type
ORDER BY total_attacks DESC
LIMIT 5;
"""

pd.read_sql(query, conn)

,threat_type,total_attacks
0,Ransomware,64
1,Malware,57
2,DDoS,52
3,SQL Injection,46
4,Phishing,42


**Prompt Used for High Severity Country Analysis**

"Identify countries with the highest number of high severity cyber attacks."



In [92]:
query = """
SELECT target_system, COUNT(*) AS high_severity_attacks
FROM cyber_threats
WHERE severity = 'High'
GROUP BY target_system
ORDER BY high_severity_attacks DESC;
"""

pd.read_sql(query, conn)

,target_system,high_severity_attacks
0,Internal Research,14
1,Financial,12
2,Human Resources,11
3,External Web,11
4,Infrastructure,8
5,Customer Database,5


In [89]:
print(pd.read_sql("PRAGMA table_info(cyber_threats);", conn))

   cid              name     type  notnull dflt_value  pk
0    0         threat_id     TEXT        0       None   0
1    1         timestamp     TEXT        0       None   0
2    2       threat_type     TEXT        0       None   0
3    3          severity     TEXT        0       None   0
4    4            status     TEXT        0       None   0
5    5     target_system     TEXT        0       None   0
6    6  severity_numeric  INTEGER        0       None   0


In [94]:
query = """
SELECT target_system, threat_type, COUNT(*) AS attack_count
FROM cyber_threats
GROUP BY target_system, threat_type
ORDER BY attack_count DESC;
"""

pd.read_sql(query, conn)


,target_system,threat_type,attack_count
0,Internal Research,Ransomware,15
1,Human Resources,Ransomware,13
2,Customer Database,DDoS,12
3,Customer Database,Ransomware,12
4,Financial,Malware,12
5,Infrastructure,Malware,12
6,Human Resources,Phishing,11
7,Internal Research,DDoS,11
8,External Web,Malware,10
9,External Web,SQL Injection,10


In [96]:
query = """
SELECT status, COUNT(*) AS detected_attacks
FROM cyber_threats
GROUP BY status
ORDER BY detected_attacks DESC;
"""

pd.read_sql(query, conn)

,status,detected_attacks
0,Detected,93
1,False Positive,80
2,Active,66
3,Mitigated,61


In [98]:
query = """
SELECT threat_type,
       ROUND(AVG(severity_numeric), 2) AS avg_severity
FROM cyber_threats
GROUP BY threat_type
ORDER BY avg_severity DESC;
"""

pd.read_sql(query, conn)

,threat_type,avg_severity
0,Phishing,2.60
1,Ransomware,2.56
2,DDoS,2.52
3,Insider Threat,2.51
4,SQL Injection,2.41
5,Malware,2.40


**Prompt Used for Operating System Risk Analysis**

"Identify which operating systems are most frequently targeted by cyber attacks."

In [100]:
query = """
SELECT target_system, COUNT(*) AS total_attacks
FROM cyber_threats
GROUP BY target_system
ORDER BY total_attacks DESC;
"""

pd.read_sql(query, conn)

,target_system,total_attacks
0,Internal Research,59
1,Human Resources,52
2,Financial,51
3,Customer Database,51
4,External Web,50
5,Infrastructure,37


**Prompt Used for Attack Outcome Ratio Analysis**

"Calculate the ratio of blocked to successful cyber attacks."

In [101]:
query = """
SELECT status, COUNT(*) AS count
FROM cyber_threats
GROUP BY status;
"""
pd.read_sql(query, conn)


,status,count
0,Active,66
1,Detected,93
2,False Positive,80
3,Mitigated,61


**SQL Task Explanation**

This SQL analysis demonstrates the use of SELECT, WHERE, GROUP BY, ORDER BY, aggregate functions, and subqueries on a cybersecurity dataset. The insights derived help identify attack patterns, severity distribution, geographical risks, and affected systems.

**Prompting Declaration**

This SQL task was completed using AI-based prompting. All SQL queries and analysis steps were executed based on structured prompts documented in this notebook.